In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats


# SET PATH
BASE = Path.home() / "Dropbox" / "PhD" / "Courses" / "Econometrics I" / "PS2"
DATA = BASE / "ps1small.csv"

df = pd.read_csv(DATA)

df["lwage"] = np.log(df["wage"])

# 30 dummies
cells = [(e, a) for e in range(12, 18) for a in range(26, 31)]
dummies = []
for e, a in cells:
    name = f"d{e}{a}"
    df[name] = ((df["education"] == e) & (df["age"] == a)).astype(int)
    dummies.append(name)

# regress lwage on the dummies w/o a constant
res = sm.OLS(df["lwage"], df[dummies]).fit()

# table
est = res.params.values.reshape(6, 5).T    
se  = res.bse.values.reshape(6, 5).T

rows, labels = [], []
for i, age in enumerate(range(26, 31)):
    rows.append([f"{v:.3f}" for v in est[i]])
    rows.append([f"({v:.3f})" for v in se[i]])
    labels += [f"age {age}", ""]

table = pd.DataFrame(rows, index=labels,
                     columns=[f"educ {e}" for e in range(12, 18)])
table

,educ 12,educ 13,educ 14,educ 15,educ 16,educ 17
age 26,9.451,9.612,9.581,9.535,10.070,9.958
,(0.091),(0.192),(0.224),(0.242),(0.167),(0.483)
age 27,9.300,9.756,9.794,9.386,9.944,10.046
,(0.085),(0.192),(0.171),(0.232),(0.187),(0.224)
age 28,9.538,9.378,9.642,9.573,9.992,10.166
,(0.083),(0.175),(0.216),(0.265),(0.148),(0.296)
age 29,9.386,9.820,9.712,9.540,10.179,9.618
,(0.085),(0.224),(0.175),(0.252),(0.131),(0.418)
age 30,9.628,9.446,9.645,9.919,10.071,10.016
,(0.088),(0.203),(0.171),(0.242),(0.164),(0.374)


In [ ]:
X = df[dummies].values
n, K = X.shape
idx = {c: i for i, c in enumerate(cells)}

# R (26 x 30) and q
rows = []
for a in range(26, 31):                       
    for e in range(12, 16):
        r = np.zeros(K); r[[idx[(e, a)], idx[(e+1, a)], idx[(e+2, a)]]] = [1, -2, 1]; rows.append(r)
for e in (12, 13):                            
    for a in range(26, 29):
        r = np.zeros(K); r[[idx[(e, a)], idx[(e, a+1)], idx[(e, a+2)]]] = [1, -2, 1]; rows.append(r)
R = np.array(rows); q = np.zeros(len(R)); J = len(R)

# F test
d = R @ res.params.values - q
s2 = res.ssr / (n - K)
F = d @ np.linalg.solve(R @ np.linalg.inv(X.T @ X) @ R.T, d) / J / s2
p = stats.f.sf(F, J, n - K)
crit = stats.f.ppf(0.95, J, n - K)

reject = F > crit
print(f"We obtain F = {F:.3f} (p = {p:.4f}), against a 5% critical value of {crit:.3f}. "
      f"We therefore {'reject' if reject else 'do not reject'} H0: the bilinear specification "
      f"{'does not' if reject else 'does'} adequately describe E(lwage | educ, age).")

We obtain F = 0.920 (p = 0.5806), against a 5% critical value of 1.508. We therefore do not reject H0: the bilinear specification does adequately describe E(lwage | educ, age).
